# Lab L2.2 · ATM cash-demand forecasting — univariate, multivariate and quantile
**Horizon Gulf Bank · Project MIZAN** · CAIP AIP-210 · Certified AI Practitioner · Day 2 · ⏱ 85 min · **Level 2 · Guided starter (fill in the blanks) · ★ recommended level**

Forecast network and per-ATM cash demand with baselines, Holt-Winters, SARIMAX with exogenous calendar drivers, and a direct 7-day-ahead ML model — then turn it into a P90 loading plan.

### How this notebook works
- Every `✏️ BLANK n` marks code **you** write. Inline blanks look like `____`; block blanks say `# ...your code here...`.
- The markdown cell above each code cell explains **what** to write, **where**, and **why**.
- Stuck > 5 min? Open the 💡 hint in the HTML lab guide, then ask your trainer. The Level 3 solution is released after the lab.

### Scenario
**Horizon Gulf Bank (HGB)** is a fictional UAE retail bank with 250,000 customers. Its Board approved **Project MIZAN** — an AI decisioning platform covering credit, fraud, collateral valuation, cash logistics, customer retention and document automation. You are a junior ML practitioner in the MIZAN squad.

The Cash Operations team loads each ATM once a week. Today they use *last month’s average* — causing ≈ 4 emergency replenishments a week and ≈ AED 40M of idle cash. You will deliver (a) a 4-week network forecast for treasury funding and (b) a 7-day-ahead per-ATM forecast with a P90 upper bound for the loading plan.

**Progressive architecture:** your cleaned network series is the input to the LSTM in Lab L4.2.

In [ ]:
# ── Project MIZAN course setup ─ run this cell first ──────────────────────────
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the course root (the folder that contains data/generate_data.py) from wherever this notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "generate_data.py").exists())
DATA = ROOT / "data"          # raw synthetic datasets (read-only)
CKPT = ROOT / "checkpoints"   # hand-off artefacts between labs (lets you start any lab fresh)
CKPT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
RNG = 42
print("Course root:", ROOT)

## Step 1 · Load the panel and repair outage gaps

**Why this matters.** On outage days the ATM was *down*, so the recorded value is missing — not zero demand. If we leave NaNs, every lag that touches them becomes NaN; if we fill with 0, the model learns fake crashes. We **interpolate for feature-building** but **exclude outage days from evaluation**.

#### ✏️ What to write in the next cell
- **Blank 1** *(inline)* — linear fill inside each ATM’s own series

In [ ]:
atm = pd.read_csv(DATA / "atm_cash_daily.csv", parse_dates=["date"]).sort_values(["atm_id", "date"]).reset_index(drop=True)
atm["was_outage"] = atm.withdrawal_aed.isna().astype(int)
print("outage ATM-days:", atm.was_outage.sum())
atm["withdrawal_aed"] = atm.groupby("atm_id").withdrawal_aed.transform(____)  # ✏️ B1
CAL = ["is_ramadan", "is_public_holiday", "is_pre_eid_window", "is_payday_window", "is_weekend", "weekend_regime", "temperature_c"]
net = atm.groupby("date").agg(withdrawal_aed=("withdrawal_aed", "sum"), **{c: (c, "first") for c in CAL}).asfreq("D")
assert net.withdrawal_aed.notna().all()
net.to_parquet(CKPT / "atm_network_daily.parquet")
net.tail()

**✔ Checkpoint.** ≈ 900 outage ATM-days; the network series is complete with 2,496 days.

## Step 2 · Univariate baselines on a 4-week horizon

**Why this matters.** Treasury needs a **28-day** network forecast. Forecast origin: 31 Aug 2025; horizon: 1–28 Sep. Baselines define the bar: *naive* (repeat the last day), *seasonal naive* (repeat the last week) and the 28-day moving average. **WAPE** (Σ|error| / Σ actual) is the metric cash teams use because it weights big days properly.

#### ✏️ What to write in the next cell
- **Blank 2** *(inline)* — weighted absolute % error
- **Blank 3** *(inline)* — repeat the last observed week 4 times

In [ ]:
origin, H = pd.Timestamp("2025-08-31"), 28
hist = net.loc[:origin, "withdrawal_aed"]
fut = net.loc[origin + pd.Timedelta(days=1): origin + pd.Timedelta(days=H)]

def fc_metrics(a, f) -> dict:
    a, f = np.asarray(a), np.asarray(f)
    return {"MAE": np.mean(np.abs(a - f)), "MAPE": np.mean(np.abs(a - f) / a), "WAPE": ____}  # ✏️ B2

fc = pd.DataFrame(index=fut.index)
fc["naive"] = hist.iloc[-1]
fc["seasonal_naive"] = ____  # ✏️ B3
fc["moving_avg_28"] = hist.iloc[-28:].mean()
pd.DataFrame({k: fc_metrics(fut.withdrawal_aed, fc[k]) for k in fc}).T.round(3)

**✔ Checkpoint.** Surprise: the 28-day moving average wins (WAPE ≈ 9–10%) and **seasonal naive is worst (≈ 17%)** — the last observed week (25–31 Aug) was a *payday* week, so repeating it over-forecasts all of September. Baselines must be chosen with the calendar in mind.

## Step 3 · Holt-Winters (univariate) and SARIMAX with calendar drivers (multivariate)

**Why this matters.** **Holt-Winters** models level + trend + weekly seasonality from the series alone. **SARIMAX** adds autoregressive structure *and* **exogenous regressors** — payday, pre-Eid, holiday and weekend flags that are known in advance. That is multivariate forecasting with only legitimate future information. We train on the post-2022 regime (relevant history).

#### ✏️ What to write in the next cell
- **Blank 4** *(inline)* — additive trend
- **Blank 5** *(inline)* — weekly cycle
- **Blank 6** *(code block, ≈2 lines)* — fit SARIMAX on log(recent) with ex_hist as exog, then forecast H steps with ex_fut and back-transform with np.exp

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
recent = hist.loc["2022-01-01":]
hw = ExponentialSmoothing(recent, trend=____, seasonal="mul", seasonal_periods=____).fit()  # ✏️ B4; B5
fc["holt_winters"] = hw.forecast(H).to_numpy()

EXOG = ["is_payday_window", "is_pre_eid_window", "is_public_holiday", "is_ramadan"]   # weekly pattern is handled by the seasonal (…,7) term
ex_hist, ex_fut = net.loc[recent.index, EXOG].astype(float), net.loc[fut.index, EXOG].astype(float)
# ✏️ BLANK 6 — write ≈2 line(s): fit SARIMAX on log(recent) with ex_hist as exog, then forecast H steps with ex_fut and back-transform with np.exp
# ...your code here...
print(sar.summary().tables[1])
display(pd.DataFrame({k: fc_metrics(fut.withdrawal_aed, fc[k]) for k in fc}).T.sort_values("WAPE").round(3))
ax = (fut.withdrawal_aed / 1e6).plot(lw=2.5, color="k", label="actual", figsize=(12, 4))
(fc[["seasonal_naive", "sarimax_exog"]] / 1e6).plot(ax=ax, style=["--", "-"]); ax.set_ylabel("AED m"); ax.legend(); plt.show()

**✔ Checkpoint.** SARIMAX with calendar drivers wins by a mile (WAPE ≈ 2%). Holt-Winters (≈ 15%) cannot know the payday ramp is coming. The payday coefficient ≈ 0.23 on the log scale ⇒ ≈ +26% withdrawals in the payday window; pre-Eid ≈ +35%.

## Step 4 · Per-ATM features for a direct 7-day-ahead model

**Why this matters.** Loading plans are per ATM, a week ahead. In a **direct** strategy every feature must be available 7 days before the target date, so the freshest lag is **lag 7**. We add calendar features known in advance, the location type, and a **day-of-week × weekend-regime** interaction so one model handles both weekend regimes.

#### ✏️ What to write in the next cell
- **Blank 7** *(inline)* — value L days earlier, within the same ATM
- **Blank 8** *(code block, ≈2 lines)* — rolling means computed only on values at least 7 days old: shift(7) first, then rolling(7) and rolling(28) means (use transform)

In [ ]:
H7 = 7
g = atm.groupby("atm_id").withdrawal_aed
for L in [7, 14, 21, 28]:
    atm[f"lag_{L}"] = ____  # ✏️ B7
# ✏️ BLANK 8 — write ≈2 line(s): rolling means computed only on values at least 7 days old: shift(7) first, then rolling(7) and rolling(28) means (use transform)
# ...your code here...
atm["dow_regime"] = atm.date.dt.day_name().str[:3] + "_" + atm.weekend_regime
atm["month"] = atm.date.dt.month
chk = atm[atm.atm_id == "ATM100"].reset_index(drop=True)
assert np.allclose(chk.lag_7.iloc[30:], chk.withdrawal_aed.shift(7).iloc[30:])   # lag_7 really is 7 days old

LAGS = ["lag_7", "lag_14", "lag_21", "lag_28", "roll7_lag7", "roll28_lag7"]
panel = atm[atm.date >= "2021-07-01"].dropna(subset=LAGS).copy()
panel[LAGS] = np.log1p(panel[LAGS]); panel["y"] = np.log1p(panel.withdrawal_aed)
panel = panel.sort_values("date").reset_index(drop=True)
test_start = pd.Timestamp("2025-07-03")
trainp, testp = panel[panel.date < test_start], panel[panel.date >= test_start]
print(len(trainp), len(testp))

**✔ Checkpoint.** ≈ 90k training and 7.2k test ATM-days (last 120 days).

## Step 5 · Linear vs gradient-boosting forecasters with TimeSeriesSplit

**Why this matters.** We compare an interpretable **Ridge** model with a **HistGradientBoosting** model (a tree ensemble you’ll study tomorrow) on identical features, tuning on `TimeSeriesSplit` folds of the training period, and benchmark both against the per-ATM seasonal naive (`lag_7`).

#### ✏️ What to write in the next cell
- **Blank 9** *(inline)* — forward-chaining folds on date-sorted rows

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
FLAGS = ["is_ramadan", "is_public_holiday", "is_pre_eid_window", "is_payday_window", "is_weekend"]
CATS = ["dow_regime", "location_type", "atm_id", "month"]
prep = ColumnTransformer([("lags", StandardScaler(), LAGS),
                          ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATS),
                          ("flags", "passthrough", FLAGS)])
ridge = GridSearchCV(Pipeline([("prep", prep), ("m", Ridge())]), {"m__alpha": [0.1, 1, 10, 100]},
                     cv=____, scoring="neg_mean_absolute_error").fit(trainp, trainp.y)  # ✏️ B9
hgb = Pipeline([("prep", prep), ("m", HistGradientBoostingRegressor(max_iter=400, learning_rate=0.06, random_state=RNG))]).fit(trainp, trainp.y)
print("ridge alpha:", ridge.best_params_)

ev = testp[testp.was_outage == 0].copy()
ev["seasonal_naive"] = np.expm1(ev.lag_7)
ev["ridge"] = np.expm1(ridge.predict(ev)); ev["hgb"] = np.expm1(hgb.predict(ev))
wape = lambda d, c: np.abs(d.withdrawal_aed - d[c]).sum() / d.withdrawal_aed.sum()
summary = pd.DataFrame({c: [wape(ev, c)] + [wape(d, c) for _, d in ev.groupby("location_type")] for c in ["seasonal_naive", "ridge", "hgb"]},
                       index=["ALL"] + sorted(ev.location_type.unique()))
summary.style.format("{:.1%}")

**✔ Checkpoint.** Both ML models halve the error of seasonal naive (≈ 17% → ≈ 8% WAPE; HGB slightly ahead). Industrial-area ATMs, driven by payday, improve the most (≈ 27% → ≈ 8–12%).

## Step 6 · P90 quantile forecasts → the loading plan

**Why this matters.** Running out of cash is far more expensive than carrying a little extra. Operations therefore load to an **upper quantile**. A model trained with **quantile loss** at 0.9 should be exceeded on only ~10% of days. Coverage, not accuracy, is the KPI here.

#### ✏️ What to write in the next cell
- **Blank 10** *(inline)* — pinball loss
- **Blank 11** *(inline)* — target exceedance rate 10%

In [ ]:
q90 = Pipeline([("prep", prep), ("m", HistGradientBoostingRegressor(loss=____, quantile=____,  # ✏️ B10; B11
                                                                     max_iter=400, learning_rate=0.06, random_state=RNG))]).fit(trainp, trainp.y)
ev["p90"] = np.expm1(q90.predict(ev))
coverage = (ev.withdrawal_aed <= ev.p90).mean()
buffer = (ev.p90 - ev.withdrawal_aed).clip(lower=0).mean()
print(f"P90 coverage: {coverage:.1%}  (target 90%) | average daily buffer per ATM: AED {buffer:,.0f}")
print(ev.groupby("location_type").apply(lambda d: (d.withdrawal_aed <= d.p90).mean()).round(3))
ev[["date", "atm_id", "location_type", "withdrawal_aed", "ridge", "hgb", "p90"]].to_parquet(CKPT / "atm_forecasts_test.parquet", index=False)

**✔ Checkpoint.** Coverage close to 90% overall (≈ 87–92%), with some variation by location type.

## ✅ Lab complete
You delivered a treasury forecast and a P90 loading plan. **Tomorrow:** classification — the heart of credit and fraud.